# Ungraded Lab: Walkthrough of ML Metadata

Keeping records at each stage of the project is an important aspect of machine learning pipelines. Especially in production models which involve many iterations of datasets and re-training, having these records will help in maintaining or debugging the deployed system. [ML Metadata](https://www.tensorflow.org/tfx/guide/mlmd) addresses this need by having an API suited specifically for keeping track of any progress made in ML projects.

In this notebook, you will look more closely at how ML Metadata can be used directly for recording and retrieving metadata independent from a TFX pipeline. You will use TFDV to infer a schema and record all information about this process, then extend the pipeline with anomaly detection, model training, and model evaluation — all tracked through MLMD.

Let's get to it!

## Imports

In [ ]:
from ml_metadata.metadata_store import metadata_store
from ml_metadata.proto import metadata_store_pb2

import tensorflow as tf
print('TF version: {}'.format(tf.__version__))

import tensorflow_data_validation as tfdv
print('TFDV version: {}'.format(tfdv.version.__version__))

import pandas as pd
import os

## Download dataset

You will be using the [Digits](https://scikit-learn.org/stable/datasets/toy_dataset.html#optical-recognition-of-handwritten-digits-dataset) dataset from scikit-learn for this lab. The dataset contains 1,797 samples of 8x8 pixel handwritten digit images (0–9), represented as 64 numerical features plus a target label. We'll load the data, and split it into train/eval/serving sets.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# Load the Digits dataset
digits = load_digits()
X = pd.DataFrame(digits.data, columns=[f'pixel_{i}' for i in range(64)])
y = pd.Series(digits.target, name='target')
data = pd.concat([X, y], axis=1)

# Split: 60% train, 20% eval, 20% serving
train, remainder = train_test_split(data, train_size=0.6, random_state=42)
eval_data, serving = train_test_split(remainder, test_size=0.5, random_state=42)

# Save splits to CSV
for split_name, split_df in [('train', train), ('eval', eval_data), ('serving', serving)]:
    split_path = os.path.join('data', split_name)
    os.makedirs(split_path, exist_ok=True)
    split_df.to_csv(os.path.join(split_path, 'data.csv'), index=False)

print(f'Digits dataset split: {len(train)} train, {len(eval_data)} eval, {len(serving)} serving')
print('\nHere\'s what we have:')
!ls -R data

## Process Outline

Here is the figure shown in class that describes the different components in an ML Metadata store:

<img src='img/mlmd_overview.png' alt='image of mlmd overview'>

The green box in the middle shows the data model followed by ML Metadata. The [official documentation](https://www.tensorflow.org/tfx/guide/mlmd#data_model) describe each of these and we'll show it here as well for easy reference:

* `ArtifactType` describes an artifact's type and its properties that are stored in the metadata store.
* An `Artifact` describes a specific instance of an ArtifactType, and its properties that are written to the metadata store.
* An `ExecutionType` describes a type of component or step in a workflow, and its runtime parameters.
* An `Execution` is a record of a component run or a step in an ML workflow and the runtime parameters.
* An `Event` is a record of the relationship between artifacts and executions.
* A `ContextType` describes a type of conceptual group of artifacts and executions in a workflow.
* A `Context` is an instance of a ContextType. It captures the shared information within the group.
* An `Attribution` is a record of the relationship between artifacts and contexts.
* An `Association` is a record of the relationship between executions and contexts.

This lab implements a **4-stage pipeline** fully tracked through MLMD:

1. **Data Validation** — Infer a schema from training data using TFDV
2. **Anomaly Detection** — Validate eval data against the schema
3. **Model Training** — Train a KNN classifier on the Digits dataset
4. **Model Evaluation** — Compute accuracy, F1, precision, and recall metrics

## Define ML Metadata's Storage Database

The first step would be to instantiate your storage backend. Here, you will use a **SQLite database** so that the metadata persists across sessions.

In [ ]:
# Instantiate a connection config with a SQLite backend for persistent storage
connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.sqlite.filename_uri = './metadata/mlmd.sqlite'

# Ensure the directory exists
os.makedirs('./metadata', exist_ok=True)

# Remove existing DB to start fresh each run
if os.path.exists('./metadata/mlmd.sqlite'):
    os.remove('./metadata/mlmd.sqlite')

# Setup the metadata store
store = metadata_store.MetadataStore(connection_config)

print(f'Metadata store created at: ./metadata/mlmd.sqlite')

## Register ArtifactTypes

Next, you will create the artifact types needed and register them to the store. You will create types for: datasets, statistics, schema, anomalies, trained model, and evaluation metrics.

In [ ]:
# Create ArtifactType for statistics
statistics_artifact_type = metadata_store_pb2.ArtifactType()
statistics_artifact_type.name = 'statistics'
statistics_artifact_type.properties['name'] = metadata_store_pb2.STRING
statistics_artifact_type.properties['split'] = metadata_store_pb2.STRING
statistics_artifact_type.properties['version'] = metadata_store_pb2.STRING

statistics_artifact_type_id = store.put_artifact_type(statistics_artifact_type)

In [ ]:
# Create ArtifactType for the input dataset
data_artifact_type = metadata_store_pb2.ArtifactType()
data_artifact_type.name = 'DataSet'
data_artifact_type.properties['name'] = metadata_store_pb2.STRING
data_artifact_type.properties['split'] = metadata_store_pb2.STRING
data_artifact_type.properties['version'] = metadata_store_pb2.INT

data_artifact_type_id = store.put_artifact_type(data_artifact_type)

# Create ArtifactType for Schema
schema_artifact_type = metadata_store_pb2.ArtifactType()
schema_artifact_type.name = 'Schema'
schema_artifact_type.properties['name'] = metadata_store_pb2.STRING
schema_artifact_type.properties['version'] = metadata_store_pb2.INT

schema_artifact_type_id = store.put_artifact_type(schema_artifact_type)

# Create ArtifactType for Anomalies
anomaly_artifact_type = metadata_store_pb2.ArtifactType()
anomaly_artifact_type.name = 'Anomalies'
anomaly_artifact_type.properties['name'] = metadata_store_pb2.STRING
anomaly_artifact_type.properties['num_anomalies'] = metadata_store_pb2.INT
anomaly_artifact_type.properties['description'] = metadata_store_pb2.STRING

anomaly_artifact_type_id = store.put_artifact_type(anomaly_artifact_type)

# Create ArtifactType for a trained Model
model_artifact_type = metadata_store_pb2.ArtifactType()
model_artifact_type.name = 'Model'
model_artifact_type.properties['name'] = metadata_store_pb2.STRING
model_artifact_type.properties['version'] = metadata_store_pb2.INT
model_artifact_type.properties['framework'] = metadata_store_pb2.STRING

model_artifact_type_id = store.put_artifact_type(model_artifact_type)

# Create ArtifactType for Model Evaluation metrics
eval_artifact_type = metadata_store_pb2.ArtifactType()
eval_artifact_type.name = 'ModelEvaluation'
eval_artifact_type.properties['name'] = metadata_store_pb2.STRING
eval_artifact_type.properties['accuracy'] = metadata_store_pb2.DOUBLE
eval_artifact_type.properties['f1_score'] = metadata_store_pb2.DOUBLE
eval_artifact_type.properties['precision'] = metadata_store_pb2.DOUBLE
eval_artifact_type.properties['recall'] = metadata_store_pb2.DOUBLE

eval_artifact_type_id = store.put_artifact_type(eval_artifact_type)

print('Data artifact type ID:', data_artifact_type_id)
print('Schema artifact type ID:', schema_artifact_type_id)
print('Anomaly artifact type ID:', anomaly_artifact_type_id)
print('Statistics artifact type ID:', statistics_artifact_type_id)
print('Model artifact type ID:', model_artifact_type_id)
print('ModelEvaluation artifact type ID:', eval_artifact_type_id)

## Register ExecutionTypes

You will create execution types for all four pipeline stages: Data Validation, Anomaly Detection, Model Training, and Model Evaluation.

In [ ]:
# Data Validation execution type
dv_execution_type = metadata_store_pb2.ExecutionType()
dv_execution_type.name = 'Data Validation'
dv_execution_type.properties['state'] = metadata_store_pb2.STRING
dv_execution_type_id = store.put_execution_type(dv_execution_type)

# Anomaly Detection execution type
anomaly_execution_type = metadata_store_pb2.ExecutionType()
anomaly_execution_type.name = 'Anomaly Detection'
anomaly_execution_type.properties['state'] = metadata_store_pb2.STRING
anomaly_execution_type_id = store.put_execution_type(anomaly_execution_type)

# Model Training execution type
training_execution_type = metadata_store_pb2.ExecutionType()
training_execution_type.name = 'Model Training'
training_execution_type.properties['state'] = metadata_store_pb2.STRING
training_execution_type_id = store.put_execution_type(training_execution_type)

# Model Evaluation execution type
eval_execution_type = metadata_store_pb2.ExecutionType()
eval_execution_type.name = 'Model Evaluation'
eval_execution_type.properties['state'] = metadata_store_pb2.STRING
eval_execution_type_id = store.put_execution_type(eval_execution_type)

print('Data validation execution type ID:', dv_execution_type_id)
print('Anomaly detection execution type ID:', anomaly_execution_type_id)
print('Model training execution type ID:', training_execution_type_id)
print('Model evaluation execution type ID:', eval_execution_type_id)

## Stage 1: Data Validation

### Generate input artifact unit

In [ ]:
# Declare input artifact of type DataSet
data_artifact = metadata_store_pb2.Artifact()
data_artifact.uri = './data/train/data.csv'
data_artifact.type_id = data_artifact_type_id
data_artifact.properties['name'].string_value = 'Digits dataset'
data_artifact.properties['split'].string_value = 'train'
data_artifact.properties['version'].int_value = 1

data_artifact_id = store.put_artifacts([data_artifact])[0]

print('Data artifact:\n', data_artifact)
print('Data artifact ID:', data_artifact_id)

### Generate execution unit and register input event

In [ ]:
# Register the Execution of a Data Validation run
dv_execution = metadata_store_pb2.Execution()
dv_execution.type_id = dv_execution_type_id
dv_execution.properties['state'].string_value = 'RUNNING'

dv_execution_id = store.put_executions([dv_execution])[0]

# Declare the input event
input_event = metadata_store_pb2.Event()
input_event.artifact_id = data_artifact_id
input_event.execution_id = dv_execution_id
input_event.type = metadata_store_pb2.Event.DECLARED_INPUT

store.put_events([input_event])

print('Data validation execution ID:', dv_execution_id)
print('Input event:\n', input_event)

### Run the TFDV component

In [ ]:
# Infer a schema by passing statistics to infer_schema()
train_data = './data/train/data.csv'
train_stats = tfdv.generate_statistics_from_csv(data_location=train_data)
schema = tfdv.infer_schema(statistics=train_stats)

schema_file = './schema.pbtxt'
tfdv.write_schema_text(schema, schema_file)

print("Dataset's Schema has been generated at:", schema_file)

### Generate output artifact and register output event

In [ ]:
# Declare output artifact of type Schema
schema_artifact = metadata_store_pb2.Artifact()
schema_artifact.uri = schema_file
schema_artifact.type_id = schema_artifact_type_id
schema_artifact.properties['version'].int_value = 1
schema_artifact.properties['name'].string_value = 'Digits Schema'

schema_artifact_id = store.put_artifacts([schema_artifact])[0]

# Declare the output event
output_event = metadata_store_pb2.Event()
output_event.artifact_id = schema_artifact_id
output_event.execution_id = dv_execution_id
output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT

store.put_events([output_event])

print('Schema artifact:\n', schema_artifact)
print('Schema artifact ID:', schema_artifact_id)

### Update the execution unit

In [ ]:
# Mark the state as COMPLETED
dv_execution.id = dv_execution_id
dv_execution.properties['state'].string_value = 'COMPLETED'

store.put_executions([dv_execution])

print('Data validation execution:\n', dv_execution)

## Stage 2: Anomaly Detection with MLMD Tracking

With a schema inferred from the training data, you can now use TFDV to check the eval data for anomalies.

In [ ]:
# Create Anomaly Detection execution and set state to RUNNING
anomaly_execution = metadata_store_pb2.Execution()
anomaly_execution.type_id = anomaly_execution_type_id
anomaly_execution.properties['state'].string_value = 'RUNNING'

anomaly_execution_id = store.put_executions([anomaly_execution])[0]

# Register eval dataset as an artifact
eval_data_for_anomaly = metadata_store_pb2.Artifact()
eval_data_for_anomaly.uri = './data/eval/data.csv'
eval_data_for_anomaly.type_id = data_artifact_type_id
eval_data_for_anomaly.properties['name'].string_value = 'Digits dataset'
eval_data_for_anomaly.properties['split'].string_value = 'eval'
eval_data_for_anomaly.properties['version'].int_value = 1

eval_data_for_anomaly_id = store.put_artifacts([eval_data_for_anomaly])[0]

# Link eval dataset as input
anomaly_eval_input = metadata_store_pb2.Event()
anomaly_eval_input.artifact_id = eval_data_for_anomaly_id
anomaly_eval_input.execution_id = anomaly_execution_id
anomaly_eval_input.type = metadata_store_pb2.Event.DECLARED_INPUT

# Link schema as input
anomaly_schema_input = metadata_store_pb2.Event()
anomaly_schema_input.artifact_id = schema_artifact_id
anomaly_schema_input.execution_id = anomaly_execution_id
anomaly_schema_input.type = metadata_store_pb2.Event.DECLARED_INPUT

store.put_events([anomaly_eval_input, anomaly_schema_input])

print('Anomaly detection execution ID:', anomaly_execution_id)
print('Eval data artifact ID:', eval_data_for_anomaly_id)

### Run TFDV anomaly detection

In [ ]:
# Generate statistics for the eval data
eval_data_path = './data/eval/data.csv'
eval_stats = tfdv.generate_statistics_from_csv(data_location=eval_data_path)

# Validate eval statistics against the inferred schema
anomalies = tfdv.validate_statistics(statistics=eval_stats, schema=schema)

# Save anomalies report
anomalies_path = './anomalies.pbtxt'
with open(anomalies_path, 'w') as f:
    f.write(str(anomalies))

# Count and summarize anomalies
anomaly_dict = dict(anomalies.anomaly_info)
num_anomalies = len(anomaly_dict)

if num_anomalies > 0:
    print(f'Found {num_anomalies} anomalies in eval data:')
    for feature_name, anomaly_info in anomaly_dict.items():
        print(f'  - {feature_name}: {anomaly_info.short_description}')
else:
    print('No anomalies found — eval data conforms to the schema.')

print(f'\nAnomalies report saved to: {anomalies_path}')

### Record anomaly results and complete execution

In [ ]:
# Build description string
if num_anomalies > 0:
    anomaly_desc = '; '.join([f'{k}: {v.short_description}' for k, v in anomaly_dict.items()])
else:
    anomaly_desc = 'No anomalies detected'

# Create the Anomalies artifact
anomaly_artifact = metadata_store_pb2.Artifact()
anomaly_artifact.uri = anomalies_path
anomaly_artifact.type_id = anomaly_artifact_type_id
anomaly_artifact.properties['name'].string_value = 'Digits Eval Anomalies'
anomaly_artifact.properties['num_anomalies'].int_value = num_anomalies
anomaly_artifact.properties['description'].string_value = anomaly_desc

anomaly_artifact_id = store.put_artifacts([anomaly_artifact])[0]

# Register output event
anomaly_output_event = metadata_store_pb2.Event()
anomaly_output_event.artifact_id = anomaly_artifact_id
anomaly_output_event.execution_id = anomaly_execution_id
anomaly_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT

store.put_events([anomaly_output_event])

# Mark as COMPLETED
anomaly_execution.id = anomaly_execution_id
anomaly_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([anomaly_execution])

print('Anomaly artifact ID:', anomaly_artifact_id)
print('Anomaly detection execution state:', anomaly_execution.properties['state'].string_value)

## Stage 3: Model Training with MLMD Tracking

Now you will train a KNN classifier on the Digits dataset and track it through MLMD.

In [ ]:
# Create a Model Training execution
training_execution = metadata_store_pb2.Execution()
training_execution.type_id = training_execution_type_id
training_execution.properties['state'].string_value = 'RUNNING'

training_execution_id = store.put_executions([training_execution])[0]

# Register the dataset as input to the training execution
training_input_event = metadata_store_pb2.Event()
training_input_event.artifact_id = data_artifact_id
training_input_event.execution_id = training_execution_id
training_input_event.type = metadata_store_pb2.Event.DECLARED_INPUT

store.put_events([training_input_event])

print('Training execution ID:', training_execution_id)

### Train a KNN classifier

In [ ]:
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Load the training data
train_df = pd.read_csv('./data/train/data.csv')
X_train = train_df.drop('target', axis=1)
y_train = train_df['target']

# Scale features (critical for KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train a KNN classifier
model = KNeighborsClassifier(n_neighbors=5, weights='distance')
model.fit(X_train_scaled, y_train)

# Save the trained model and scaler
model_path = './model/model.pkl'
scaler_path = './model/scaler.pkl'
os.makedirs('./model', exist_ok=True)
with open(model_path, 'wb') as f:
    pickle.dump(model, f)
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f'Model trained on {len(X_train)} samples with {X_train.shape[1]} features')
print(f'Training accuracy: {model.score(X_train_scaled, y_train):.4f}')
print(f'Model saved to: {model_path}')

### Record the trained model artifact and complete the execution

In [ ]:
# Create the trained model artifact
model_artifact = metadata_store_pb2.Artifact()
model_artifact.uri = model_path
model_artifact.type_id = model_artifact_type_id
model_artifact.properties['name'].string_value = 'Digits KNN'
model_artifact.properties['version'].int_value = 1
model_artifact.properties['framework'].string_value = 'scikit-learn'

model_artifact_id = store.put_artifacts([model_artifact])[0]

# Register output event
training_output_event = metadata_store_pb2.Event()
training_output_event.artifact_id = model_artifact_id
training_output_event.execution_id = training_execution_id
training_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT

store.put_events([training_output_event])

# Mark as COMPLETED
training_execution.id = training_execution_id
training_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([training_execution])

print('Model artifact ID:', model_artifact_id)
print('Training execution state:', training_execution.properties['state'].string_value)

## Stage 4: Model Evaluation with MLMD Tracking

Evaluate the model on the held-out eval dataset and record the metrics.

In [ ]:
# Register the eval dataset artifact
eval_data_artifact = metadata_store_pb2.Artifact()
eval_data_artifact.uri = './data/eval/data.csv'
eval_data_artifact.type_id = data_artifact_type_id
eval_data_artifact.properties['name'].string_value = 'Digits dataset'
eval_data_artifact.properties['split'].string_value = 'eval'
eval_data_artifact.properties['version'].int_value = 1

eval_data_artifact_id = store.put_artifacts([eval_data_artifact])[0]

# Create Model Evaluation execution
eval_execution = metadata_store_pb2.Execution()
eval_execution.type_id = eval_execution_type_id
eval_execution.properties['state'].string_value = 'RUNNING'

eval_execution_id = store.put_executions([eval_execution])[0]

# Register input events: model + eval dataset
model_input_event = metadata_store_pb2.Event()
model_input_event.artifact_id = model_artifact_id
model_input_event.execution_id = eval_execution_id
model_input_event.type = metadata_store_pb2.Event.DECLARED_INPUT

eval_data_input_event = metadata_store_pb2.Event()
eval_data_input_event.artifact_id = eval_data_artifact_id
eval_data_input_event.execution_id = eval_execution_id
eval_data_input_event.type = metadata_store_pb2.Event.DECLARED_INPUT

store.put_events([model_input_event, eval_data_input_event])

print('Eval dataset artifact ID:', eval_data_artifact_id)
print('Evaluation execution ID:', eval_execution_id)

### Run evaluation and compute metrics

In [ ]:
import json
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Load the eval data
eval_df = pd.read_csv('./data/eval/data.csv')
X_eval = eval_df.drop('target', axis=1)
y_eval = eval_df['target']

# Scale using the same scaler from training
X_eval_scaled = scaler.transform(X_eval)

# Generate predictions
y_pred = model.predict(X_eval_scaled)

# Compute metrics
metrics = {
    'accuracy': accuracy_score(y_eval, y_pred),
    'f1_score': f1_score(y_eval, y_pred, average='weighted'),
    'precision': precision_score(y_eval, y_pred, average='weighted'),
    'recall': recall_score(y_eval, y_pred, average='weighted')
}

# Save metrics to JSON
metrics_path = './model/eval_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Evaluation on {len(X_eval)} samples:')
for metric_name, value in metrics.items():
    print(f'  {metric_name}: {value:.4f}')
print(f'\nMetrics saved to: {metrics_path}')

### Record the evaluation metrics artifact and complete the execution

In [ ]:
# Create the ModelEvaluation artifact
eval_artifact = metadata_store_pb2.Artifact()
eval_artifact.uri = metrics_path
eval_artifact.type_id = eval_artifact_type_id
eval_artifact.properties['name'].string_value = 'Digits KNN Evaluation'
eval_artifact.properties['accuracy'].double_value = metrics['accuracy']
eval_artifact.properties['f1_score'].double_value = metrics['f1_score']
eval_artifact.properties['precision'].double_value = metrics['precision']
eval_artifact.properties['recall'].double_value = metrics['recall']

eval_artifact_id = store.put_artifacts([eval_artifact])[0]

# Register output event
eval_output_event = metadata_store_pb2.Event()
eval_output_event.artifact_id = eval_artifact_id
eval_output_event.execution_id = eval_execution_id
eval_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT

store.put_events([eval_output_event])

# Mark as COMPLETED
eval_execution.id = eval_execution_id
eval_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([eval_execution])

print('Evaluation artifact ID:', eval_artifact_id)
print('Evaluation execution state:', eval_execution.properties['state'].string_value)

## Setting up Context Types and Generating a Context Unit

You can group all artifacts and executions into a `Context`.

In [ ]:
# Create a ContextType
expt_context_type = metadata_store_pb2.ContextType()
expt_context_type.name = 'Experiment'
expt_context_type.properties['note'] = metadata_store_pb2.STRING

expt_context_type_id = store.put_context_type(expt_context_type)

# Generate the context
expt_context = metadata_store_pb2.Context()
expt_context.type_id = expt_context_type_id
expt_context.name = 'Digits Classification Pipeline'
expt_context.properties['note'].string_value = 'Data validation, anomaly detection, model training, and evaluation for Digits dataset'

expt_context_id = store.put_contexts([expt_context])[0]

print('Experiment Context type ID:', expt_context_type_id)
print('Experiment Context ID:', expt_context_id)

## Generate attribution and association relationships

In [ ]:
# Attributions for schema, anomalies, model, and evaluation artifacts
schema_attribution = metadata_store_pb2.Attribution()
schema_attribution.artifact_id = schema_artifact_id
schema_attribution.context_id = expt_context_id

anomaly_attribution = metadata_store_pb2.Attribution()
anomaly_attribution.artifact_id = anomaly_artifact_id
anomaly_attribution.context_id = expt_context_id

model_attribution = metadata_store_pb2.Attribution()
model_attribution.artifact_id = model_artifact_id
model_attribution.context_id = expt_context_id

eval_attribution = metadata_store_pb2.Attribution()
eval_attribution.artifact_id = eval_artifact_id
eval_attribution.context_id = expt_context_id

# Associations for all four execution steps
dv_association = metadata_store_pb2.Association()
dv_association.execution_id = dv_execution_id
dv_association.context_id = expt_context_id

anomaly_association = metadata_store_pb2.Association()
anomaly_association.execution_id = anomaly_execution_id
anomaly_association.context_id = expt_context_id

training_association = metadata_store_pb2.Association()
training_association.execution_id = training_execution_id
training_association.context_id = expt_context_id

eval_association = metadata_store_pb2.Association()
eval_association.execution_id = eval_execution_id
eval_association.context_id = expt_context_id

store.put_attributions_and_associations(
    [schema_attribution, anomaly_attribution, model_attribution, eval_attribution],
    [dv_association, anomaly_association, training_association, eval_association]
)

print('Attributions: Schema, Anomalies, Model, Evaluation')
print('Associations: Data Validation, Anomaly Detection, Model Training, Model Evaluation')

## Retrieving Information from the Metadata Store

You've now recorded everything. Let's query the store to trace lineage.

In [ ]:
# Get all artifact types
store.get_artifact_types()

In [ ]:
# Investigate which dataset was used to generate the schema
schema_to_inv = store.get_artifacts_by_type('Schema')[0]
print(schema_to_inv)

In [ ]:
# Get events related to the schema
schema_events = store.get_events_by_artifact_ids([schema_to_inv.id])
print(schema_events)

In [ ]:
# Get all events for the execution that produced the schema
execution_events = store.get_events_by_execution_ids([schema_events[0].execution_id])
print(execution_events)

In [ ]:
# Look up the input artifact (dataset) that was used
artifact_input = execution_events[0]
store.get_artifacts_by_id([artifact_input.artifact_id])

### Retrieve anomaly detection results

In [ ]:
# Get the anomaly detection results
anomaly_to_inv = store.get_artifacts_by_type('Anomalies')[0]
print(f'Anomalies found: {anomaly_to_inv.properties["num_anomalies"].int_value}')
print(f'Description: {anomaly_to_inv.properties["description"].string_value}')

# Trace back to inputs
anomaly_events = store.get_events_by_artifact_ids([anomaly_to_inv.id])
anomaly_exec_events = store.get_events_by_execution_ids([anomaly_events[0].execution_id])

print('\nInputs to the anomaly detection execution:')
for event in anomaly_exec_events:
    if event.type == metadata_store_pb2.Event.DECLARED_INPUT:
        artifact = store.get_artifacts_by_id([event.artifact_id])[0]
        artifact_type = store.get_artifact_types_by_id([artifact.type_id])[0]
        print(f'  [{artifact_type.name}] {artifact.properties["name"].string_value} (uri: {artifact.uri})')

### Trace model lineage

In [ ]:
# Get the trained model artifact and trace its lineage
model_to_inv = store.get_artifacts_by_type('Model')[0]
print('Model artifact:\n', model_to_inv)

model_events = store.get_events_by_artifact_ids([model_to_inv.id])
training_exec_id = model_events[0].execution_id
training_events = store.get_events_by_execution_ids([training_exec_id])

training_input = [e for e in training_events if e.type == metadata_store_pb2.Event.DECLARED_INPUT][0]
training_dataset = store.get_artifacts_by_id([training_input.artifact_id])[0]
print('\nDataset used to train the model:')
print(training_dataset)

### Trace evaluation metrics lineage

In [ ]:
# Get evaluation metrics and trace back to inputs
eval_to_inv = store.get_artifacts_by_type('ModelEvaluation')[0]
print('Evaluation artifact metrics:')
print(f'  Accuracy:  {eval_to_inv.properties["accuracy"].double_value:.4f}')
print(f'  F1 Score:  {eval_to_inv.properties["f1_score"].double_value:.4f}')
print(f'  Precision: {eval_to_inv.properties["precision"].double_value:.4f}')
print(f'  Recall:    {eval_to_inv.properties["recall"].double_value:.4f}')

eval_events = store.get_events_by_artifact_ids([eval_to_inv.id])
eval_exec_events = store.get_events_by_execution_ids([eval_events[0].execution_id])

print('\nInputs to the evaluation execution:')
for event in eval_exec_events:
    if event.type == metadata_store_pb2.Event.DECLARED_INPUT:
        artifact = store.get_artifacts_by_id([event.artifact_id])[0]
        artifact_type = store.get_artifact_types_by_id([artifact.type_id])[0]
        print(f'  [{artifact_type.name}] {artifact.properties["name"].string_value} (uri: {artifact.uri})')

### Full pipeline summary

In [ ]:
# List all artifacts and executions in the experiment context
print('=== All artifacts in the experiment context ===')
context_artifacts = store.get_artifacts_by_context(expt_context_id)
for a in context_artifacts:
    a_type = store.get_artifact_types_by_id([a.type_id])[0]
    print(f'  [{a_type.name}] {a.properties["name"].string_value} (id: {a.id})')

print('\n=== All executions in the experiment context ===')
context_executions = store.get_executions_by_context(expt_context_id)
for e in context_executions:
    e_type = store.get_execution_types_by_id([e.type_id])[0]
    print(f'  [{e_type.name}] state: {e.properties["state"].string_value} (id: {e.id})')

### Wrap Up

In this notebook, you practiced using ML Metadata outside of TFX to track a complete ML pipeline with four stages:

1. **Data Validation** — Inferred a schema from training data using TFDV
2. **Anomaly Detection** — Validated eval data against the schema to catch data drift
3. **Model Training** — Trained a KNN classifier on the Digits dataset
4. **Model Evaluation** — Computed accuracy, F1, precision, and recall metrics

All artifacts, executions, and their relationships were recorded in a **persistent SQLite-backed** metadata store, enabling full lineage tracking across sessions.